# ENERGICAL — Exploratory Data Analysis & Cleaning
### DataFest 2026 | The outliers | June 2026


## 0. Imports & Configuration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px

## 1. Data Loading

In [ ]:

df= pd.read_csv(r"C:\Users\PCPRODZ\Downloads\LIVRABLE_DataFest2026_ENERGICAL\LIVRABLE_etudiants\energical_transactions_anonymized.csv")
print(df.info())
print(df.head())
print("Transactions:", df.shape)

In [ ]:
Catalogue =pd.read_csv(r"C:\Users\PCPRODZ\Downloads\LIVRABLE_DataFest2026_ENERGICAL\LIVRABLE_etudiants\energical_catalogue_produits.csv")
print("second df")
print(Catalogue.info())
print(Catalogue.head())
print("Catalogue:", Catalogue.shape)

### Findings: Raw data structure

**Transactions dataset:**
- 17,263 rows × 13 columns
- Key nulls: `date_commande` (16), `produit` (110), `quantite`/`montant_da` (101), `moyen_paiement` (11)
- Columns cover: customer ID, order ID, date, wilaya (location), product, quantity, amount (DZD), payment method, order status

**Catalogue dataset:**
- 731 rows × 10 columns
- Key nulls: `prix_da` (218), `description_courte` (332) — acceptable, not critical
- Columns: product ID, name, category, subcategory, price, stock status, alternative product, margin estimate, restock delay

→ Ready for cleaning.

## 2. Data Cleaning & Preprocessing

#### 2.1 Filter statuts

In [ ]:
df_excluded = df[df['statut_commande'] != 'Terminée'].copy()
df = df[df['statut_commande'] == 'Terminée'].copy()

print(f"Terminée: {len(df):,}")
print(f"Excluded: {len(df_excluded):,}")
print(df_excluded['statut_commande'].value_counts())

### Findings: Order status breakdown

- **Terminée** (completed): 16,498 orders (95.6%) which is our working dataset
- **Retour NOEST**: 233 orders
- **Partiellement remboursée**: 80 orders
- **En livraison par NOEST**: 412 orders (in transit, excluded)
- **Other** (pending, partial payment): 40 orders

**Total excluded**: 765 orders (4.4% loss)

→ Minimal data loss. Excluded data is preserved for return rate geo-analysis.

#### 2.2 Fix dtypes

In [ ]:
df["date_commande"] = pd.to_datetime(df["date_commande"], errors="coerce")
df["quantite"]      = df["quantite"].astype("Int64")   
df["montant_da"]    = df["montant_da"].astype(float)
before = len(df)

#### 2.3 Handle nulls

In [ ]:
df = df.dropna(subset=["date_commande"])
df = df.dropna(subset=["quantite", "montant_da"])
print(f"Dropped {before - len(df)} rows (missing date/revenue/Quantity)")
df["produit"] = df["produit"].fillna("Inconnu")
df["moyen_paiement"] = df["moyen_paiement"].fillna("Non renseigné")
print(f"Rows after null handling: {len(df):,}")

### Findings: Null handling complete

- Dropped 111 rows (missing date/revenue/quantity)
- Filled `produit` with 'Inconnu' (preserves order count)
- Filled `moyen_paiement` with 'Non renseigné'

**Total data loss**: 111 / 16,498 rows = 0.67%

→ Minimal impact.

### 2.4 Standardize wilayas

In [ ]:
df["wilaya"] = df["wilaya"].str.lower().str.strip()
df["wilaya"] = df["wilaya"].replace({"tebessa": "tébessa"})
df = df[~df["wilaya"].isin(["bamako", "les ulis"])]
df["wilaya"].value_counts()

In [ ]:
print(df[df["quantite"] <0])
df['montant_da'] = df['montant_da'].abs()
print(df[df["montant_da"] <0])

In [ ]:
print(df.info())
print(df.describe())
print(df.isnull().sum())
print(df['date_commande'].min(), df['date_commande'].max())

###  Findings: Wilaya standardization

- Converted wilaya's name to lowercase + stripped whitespace
- Fixed spelling like =: `tebessa` → `tébessa`
- Removed invalid entries: Bamako, Les Ulis (non-Algerian cities) which were 2 rows

**Result**: 48 unique wilayas across Algeria.

→ Geographic data clean. Ready for location analysis.

### 2.4 Cleaning catalogue and merging

In [ ]:
df['produit_key'] = df['produit'].str[:15].str.strip()
Catalogue['nom_key'] = Catalogue['nom_produit'].str[:15].str.strip()

df_merged = pd.merge(
    df, Catalogue,
    left_on='produit_key',
    right_on='nom_key',
    how='left'
)

unmatched = df_merged['nom_key'].isna().sum()
print(f"Unmatched: {unmatched}")

In [ ]:
print(f"Match rate: {(len(df_merged) - unmatched) / len(df_merged) * 100:.1f}%")

### Findings: Catalogue merge quality

- Merge strategy: first-15-character product name prefix
- **Match rate: 96.2%** (14,816 / 15,387 rows matched) ✓

→ High match rate validates strategy. Ready for RFM.

## 3. Exploratory Data Analysis (EDA)

The following charts reveal strategic patterns.

### 3.1 Monthly revenue:

In [ ]:
monthly = df_merged.groupby(df_merged['date_commande'].dt.to_period('M'))['montant_da'].sum().reset_index()
monthly['date_commande'] = monthly['date_commande'].astype(str)

fig = px.line(monthly, x='date_commande', y='montant_da',
              title="Chiffre d'affaires mensuel 2022–2024")
fig.show()

###  Insight: Monthly revenue 2022–2024

- **Clear winter seasonality**: December–February peak 
- **Summer valley**: June–August drop (seasonal pause)
- **Pattern stable across 3 years** — predictable, use for inventory planning

→ Winter = critical season. 

### 3.2 Top 10 products:


In [ ]:
top10 = df_merged.groupby('produit')['montant_da'].sum().sort_values(ascending=False).head(10).reset_index()

# shorten name to first 25 chars
top10['produit_short'] = top10['produit'].str[:25]

fig = px.bar(top10, x='montant_da', y='produit_short', orientation='h',
             title="Top 10 Products by Revenue")
fig.show()

### 3.3 Revenue by wilaya:

In [ ]:
wilaya_rev = df_merged.groupby('wilaya')['montant_da'].sum().sort_values(ascending=False).head(15).reset_index()

fig = px.bar(wilaya_rev, x='wilaya', y='montant_da',
             title="Revenue per wilaya — Top 15")
fig.show()

### Insight: Revenue by wilaya (Top 15)

- **Alger dominates**: 3,747 transactions (21.5% of total volume)
- **Tlemcen punches above weight**: 908 transactions despite smaller population
- **Geographic concentration**: Top 5 wilayas = 65% of revenue


### 3.4 B2B vs B2C:

In [ ]:
b2b_b2c = df_merged.groupby('type_client').agg(
    revenue=('montant_da', 'sum'),
    orders=('id_commande_anon', 'nunique'),
    avg_basket=('montant_da', 'mean')
).reset_index()
fig = px.bar(b2b_b2c, x='type_client', y='revenue',
             text=b2b_b2c['revenue'].apply(lambda x: f"{x/1e6:.1f}M DZD"),
             title="Revenue by Client Type")
fig.update_traces(textposition='outside')
fig.show()

###  Insight: Client type comparison

- **B2C dominates volume**: 7,687 orders vs B2B 58 orders
- **B2B = high-value segment**: Avg order 25,287 DZD vs B2C 16,997 DZD
- **B2B = 49% higher basket** despite 99% smaller order count

→ Strategic insight: B2B is tiny but profitable. 

### 3.5 revenue by product Category

In [ ]:
cat = df_merged.groupby('categorie_produit')['montant_da'].sum().sort_values(ascending=False).reset_index()

fig = px.bar(cat, x='categorie_produit', y='montant_da',
             title="Revenue by Product Category")
fig.show()

### 3.6 Returns by wilaya

In [ ]:
returns_by_wilaya = df_excluded[df_excluded['statut_commande'] == 'Retour NOEST'].groupby('wilaya').size().reset_index(name='returns')
returns_by_wilaya = returns_by_wilaya.sort_values('returns', ascending=False).head(15)

fig = px.bar(returns_by_wilaya, x='wilaya', y='returns',
             title="Returns by Wilaya")
fig.show()

## 5. Data Export

In [ ]:
df_merged = df_merged.drop_duplicates(subset=['id_transaction'])
print(f"After dedup: {len(df_merged)} rows")
df_merged.to_csv('data_clean.csv', index=False)
df_excluded.to_csv('data_returns.csv', index=False)
print(f"Exported clean: {len(df_merged)} rows")
print(f"Exported returns: {len(df_excluded)} rows")